# Coverage vs Bandwidth Trade-offs

This notebook analyzes how different compression / transmission strategies affect exploration coverage and communication efficiency in multi-robot scenarios.

## Question
Given a fixed total bandwidth budget, should robots send:
1. **Full-map uploads** periodically (high accuracy, high redundancy)
2. **Sparse differential updates** (lower redundancy, incremental)
3. **Hybrid approach** (full updates at longer intervals, diffs in between)

In [2]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUndefinedVariable=false, reportUnusedImport=false
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Setup: Define Map Update Strategies

In [3]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUndefinedVariable=false, reportUnusedImport=false
from __future__ import annotations

import numpy as np

grid_size: int = 100
total_grid_cells: int = grid_size ** 2
bandwidth_mbps: float = 2.0
update_period_s: float = 1.0
bytes_per_update: float = bandwidth_mbps * 1e6 * update_period_s / 8.0


class UpdateStrategy:
    def __init__(self, name: str) -> None:
        self.name = name

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        raise NotImplementedError

    def can_transmit(self, available_bytes: float, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> bool:
        return self.estimate_bytes(grid_state, changed_cells) <= available_bytes


class FullMapStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Full Map")

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        return total_grid_cells + 100


class SparseUpdateStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Sparse Diff")

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        return len(changed_cells) * 3 + 200


class HybridStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Hybrid (Full + Diff)")
        self.update_count = 0

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        self.update_count += 1
        if self.update_count % 10 == 0:
            return total_grid_cells + 100
        return len(changed_cells) * 3 + 200


print("Strategies defined.")

Strategies defined.


In [4]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUndefinedVariable=false, reportUnusedImport=false
from __future__ import annotations

import numpy as np

grid_size: int = 100
total_grid_cells: int = grid_size ** 2
bandwidth_mbps: float = 2.0
update_period_s: float = 1.0
bytes_per_update: float = bandwidth_mbps * 1e6 * update_period_s / 8.0


class UpdateStrategy:
    def __init__(self, name: str) -> None:
        self.name = name

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        raise NotImplementedError


class FullMapStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Full Map")

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        return total_grid_cells + 100


class SparseUpdateStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Sparse Diff")

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        return len(changed_cells) * 3 + 200


class HybridStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Hybrid (Full + Diff)")
        self.update_count = 0

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        self.update_count += 1
        if self.update_count % 10 == 0:
            return total_grid_cells + 100
        return len(changed_cells) * 3 + 200


def simulate_exploration(strategy: UpdateStrategy, sim_steps: int = 300) -> tuple[list[int], list[float], list[float]]:
    grid: np.ndarray = np.full((grid_size, grid_size), -1, dtype=int)
    coverage_history: list[float] = []
    bytes_history: list[float] = []
    times: list[int] = []
    total_bytes: float = 0.0

    for step in range(sim_steps):
        num_explore: int = int(np.random.randint(10, 30))
        explore_idx: list[int] = np.random.choice(total_grid_cells, num_explore, replace=False).tolist()
        changed_cells: list[tuple[int, int]] = []
        for idx in explore_idx:
            i: int = idx // grid_size
            j: int = idx % grid_size
            if grid[i, j] == -1:
                grid[i, j] = int(np.random.randint(0, 100))
                changed_cells.append((i, j))
        bytes_needed: int = strategy.estimate_bytes(grid, changed_cells)
        if bytes_needed <= bytes_per_update:
            total_bytes += bytes_needed
        explored: int = int(np.sum(grid != -1))
        coverage_frac: float = float(explored / total_grid_cells)
        coverage_history.append(coverage_frac)
        bytes_history.append(total_bytes / 1e6)
        times.append(step)
    return times, coverage_history, bytes_history


strategies: list[UpdateStrategy] = [FullMapStrategy(), SparseUpdateStrategy(), HybridStrategy()]
results: dict[str, tuple[list[int], list[float], list[float]]] = {}
for strat in strategies:
    times, coverage, bytes_sent = simulate_exploration(strat, sim_steps=300)
    results[strat.name] = (times, coverage, bytes_sent)
    print(f"{strat.name}: {coverage[-1]*100:.1f}% coverage, {bytes_sent[-1]:.2f} MB transmitted")

Full Map: 45.0% coverage, 3.03 MB transmitted
Sparse Diff: 44.3% coverage, 0.07 MB transmitted
Hybrid (Full + Diff): 44.1% coverage, 0.37 MB transmitted


# pyright: reportMissingImports=false, reportMissingModuleSource=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false, reportUndefinedVariable=false, reportUnusedImport=false
from __future__ import annotations

import numpy as np

grid_size: int = 100
total_grid_cells: int = grid_size ** 2
bandwidth_mbps: float = 2.0
update_period_s: float = 1.0
bytes_per_update: float = bandwidth_mbps * 1e6 * update_period_s / 8.0


class UpdateStrategy:
    def __init__(self, name: str) -> None:
        self.name = name

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        raise NotImplementedError


class FullMapStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Full Map")

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        return total_grid_cells + 100


class SparseUpdateStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Sparse Diff")

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        return len(changed_cells) * 3 + 200


class HybridStrategy(UpdateStrategy):
    def __init__(self) -> None:
        super().__init__("Hybrid (Full + Diff)")
        self.update_count = 0

    def estimate_bytes(self, grid_state: np.ndarray, changed_cells: list[tuple[int, int]]) -> int:
        self.update_count += 1
        if self.update_count % 10 == 0:
,
100
,
3
200
,
,
,